In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
)

In [2]:
df = pd.read_csv("data/df_final.csv")
df.columns

Index(['movimiento', 'repeticion_id', 'ventana', 'pitch1_std', 'pitch1_median',
       'pitch1_min', 'pitch1_max', 'pitch1_iqr', 'pitch1_mad_diff', 'yaw1_std',
       'yaw1_median', 'yaw1_min', 'yaw1_max', 'yaw1_iqr', 'yaw1_mad_diff',
       'roll1_std', 'roll1_median', 'roll1_min', 'roll1_max', 'roll1_iqr',
       'roll1_mad_diff', 'pitch2_std', 'pitch2_median', 'pitch2_min',
       'pitch2_max', 'pitch2_iqr', 'pitch2_mad_diff', 'yaw2_std',
       'yaw2_median', 'yaw2_min', 'yaw2_max', 'yaw2_iqr', 'yaw2_mad_diff',
       'roll2_std', 'roll2_median', 'roll2_min', 'roll2_max', 'roll2_iqr',
       'roll2_mad_diff', 'f1_std', 'f1_median', 'f1_min', 'f1_max', 'f1_iqr',
       'f1_mad_diff', 'f2_std', 'f2_median', 'f2_min', 'f2_max', 'f2_iqr',
       'f2_mad_diff', 'f3_std', 'f3_median', 'f3_min', 'f3_max', 'f3_iqr',
       'f3_mad_diff', 'f4_std', 'f4_median', 'f4_min', 'f4_max', 'f4_iqr',
       'f4_mad_diff', 'f5_std', 'f5_median', 'f5_min', 'f5_max', 'f5_iqr',
       'f5_mad_diff', 'pit

In [3]:
# ============================================================
# 1. Datos
# ============================================================

N_SPLITS = 5
SEED = 42

X = df.drop(
    columns=["movimiento", "repeticion_id", "ventana"]
).values

# Se usan etiquetas de texto para facilitar los reportes.
y = df["movimiento"].astype(str)
groups = df["repeticion_id"]
clases = np.sort(y.unique())

print(f"Ventanas: {len(X):,}")
print(f"Características: {X.shape[1]}")
print(f"Movimientos: {len(clases)}")
print(f"Grupos: {groups.nunique():,}")

Ventanas: 34,056
Características: 72
Movimientos: 15
Grupos: 4,257


In [4]:
# ============================================================
# 2. Pipelines
# ============================================================

def crear_pipeline(modelo, escalar=False):
    pasos = [
        ("imputacion", SimpleImputer(strategy="median")),
    ]

    if escalar:
        pasos.append(("escalamiento", StandardScaler()))

    pasos.append(("modelo", modelo))
    return Pipeline(pasos)


modelos = {
    "Dummy": crear_pipeline(
        DummyClassifier(strategy="most_frequent")
    ),

    "Extra Trees": crear_pipeline(
        ExtraTreesClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        )
    ),

    "Random Forest": crear_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        )
    ),

    "HistGradientBoosting": crear_pipeline(
        HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            # Evita una partición interna aleatoria por ventanas.
            early_stopping=False,
            random_state=SEED,
        )
    ),

    "SVM RBF": crear_pipeline(
        SVC(
            C=10,
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            cache_size=512,
        ),
        escalar=True,
    ),

    "KNN": crear_pipeline(
        KNeighborsClassifier(
            n_neighbors=7,
            weights="distance",
            metric="minkowski",
            p=2,
            n_jobs=-1,
        ),
        escalar=True,
    ),

    "Regresión logística": crear_pipeline(
        LogisticRegression(
            C=1.0,
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED,
        ),
        escalar=True,
    ),

    "LDA regularizado": crear_pipeline(
        LinearDiscriminantAnalysis(
            solver="lsqr",
            shrinkage="auto",
        ),
        escalar=True,
    ),

    "MLP": crear_pipeline(
        MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=0.01,
            learning_rate_init=0.001,
            batch_size=128,
            max_iter=400,
            # Evita validación interna que ignore los grupos.
            early_stopping=False,
            random_state=SEED,
        ),
        escalar=True,
    ),
}

In [5]:
# ============================================================
# 3. Crear los mismos folds para todos los modelos
# ============================================================

cv = GroupKFold(n_splits=N_SPLITS)
splits = list(cv.split(X, y, groups=groups))

for fold, (idx_train, idx_val) in enumerate(splits, start=1):
    grupos_train = set(groups.iloc[idx_train])
    grupos_val = set(groups.iloc[idx_val])

    print(
        f"Fold {fold}: "
        f"{len(idx_train):,} ventanas de entrenamiento / "
        f"{len(idx_val):,} de validación | "
        f"{len(grupos_train)} / {len(grupos_val)} grupos"
    )

Fold 1: 27,240 ventanas de entrenamiento / 6,816 de validación | 3405 / 852 grupos
Fold 2: 27,240 ventanas de entrenamiento / 6,816 de validación | 3405 / 852 grupos
Fold 3: 27,248 ventanas de entrenamiento / 6,808 de validación | 3406 / 851 grupos
Fold 4: 27,248 ventanas de entrenamiento / 6,808 de validación | 3406 / 851 grupos
Fold 5: 27,248 ventanas de entrenamiento / 6,808 de validación | 3406 / 851 grupos


In [6]:
# ============================================================
# 4. Evaluación y predicciones fuera de muestra (OOF)
# ============================================================

resultados_folds = []
resumen = []

reportes = {}
predicciones_oof = {}
matrices_confusion = {}

for nombre, pipeline in modelos.items():
    print(f"\n{'=' * 60}\n{nombre}\n{'=' * 60}")

    inicio = time.perf_counter()

    # Cada fila recibirá una predicción del fold donde fue validación.
    oof = np.empty(len(y), dtype=y.to_numpy().dtype)
    evaluada = np.zeros(len(y), dtype=bool)
    f1_folds = []

    for fold, (idx_train, idx_val) in enumerate(splits, start=1):
        modelo = clone(pipeline)

        modelo.fit(
            X[idx_train],
            y[idx_train],
        )

        pred = modelo.predict(X[idx_val])

        oof[idx_val] = pred
        evaluada[idx_val] = True

        accuracy = accuracy_score(y[idx_val], pred)

        f1_macro = f1_score(
            y[idx_val],
            pred,
            labels=clases,
            average="macro",
            zero_division=0,
        )

        f1_folds.append(f1_macro)

        resultados_folds.append({
            "modelo": nombre,
            "fold": fold,
            "accuracy": accuracy,
            "f1_macro": f1_macro,
            "n_validacion": len(idx_val),
        })

        print(
            f"Fold {fold}: "
            f"accuracy={accuracy:.4f} | "
            f"F1 macro={f1_macro:.4f}",
            flush=True,
        )

    assert evaluada.all()

    segundos = time.perf_counter() - inicio

    print("\nClassification report OOF:")
    print(
        classification_report(
            y,
            oof,
            labels=clases,
            digits=4,
            zero_division=0,
        )
    )

    reporte = classification_report(
        y,
        oof,
        labels=clases,
        output_dict=True,
        zero_division=0,
    )

    reportes[nombre] = pd.DataFrame(reporte).T
    predicciones_oof[nombre] = oof.copy()

    matrices_confusion[nombre] = pd.DataFrame(
        confusion_matrix(y, oof, labels=clases),
        index=pd.Index(clases, name="Real"),
        columns=pd.Index(clases, name="Predicción"),
    )

    resumen.append({
        "modelo": nombre,
        "accuracy_oof": accuracy_score(y, oof),
        "precision_macro_oof": reporte["macro avg"]["precision"],
        "recall_macro_oof": reporte["macro avg"]["recall"],
        "f1_macro_oof": reporte["macro avg"]["f1-score"],
        "f1_weighted_oof": reporte["weighted avg"]["f1-score"],
        "f1_macro_cv_media": np.mean(f1_folds),
        "f1_macro_cv_std": np.std(f1_folds, ddof=1),
        "tiempo_segundos": segundos,
    })


Dummy
Fold 1: accuracy=0.0892 | F1 macro=0.0109
Fold 2: accuracy=0.0904 | F1 macro=0.0111
Fold 3: accuracy=0.0905 | F1 macro=0.0111
Fold 4: accuracy=0.0917 | F1 macro=0.0112
Fold 5: accuracy=0.0905 | F1 macro=0.0111

Classification report OOF:
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000      1856
           1     0.0000    0.0000    0.0000      1696
          10     0.0000    0.0000    0.0000      2488
          11     0.0000    0.0000    0.0000      1880
          12     0.0000    0.0000    0.0000      2344
          13     0.0000    0.0000    0.0000      2504
          15     0.0000    0.0000    0.0000      2504
           2     0.0000    0.0000    0.0000      2136
           3     0.0000    0.0000    0.0000      2000
           4     0.0000    0.0000    0.0000      2296
           5     0.0000    0.0000    0.0000      2344
           6     0.0000    0.0000    0.0000      2080
           7     0.0904    1.0000    0.1659      308

In [7]:
# ============================================================
# 5. Comparación final
# ============================================================

df_folds = pd.DataFrame(resultados_folds)

df_resultados = (
    pd.DataFrame(resumen)
    .sort_values("f1_macro_cv_media", ascending=False)
    .reset_index(drop=True)
)

print("\nCOMPARACIÓN DE MODELOS")
print(df_resultados.round(4).to_string(index=False))

# Objetos disponibles:
# df_resultados                    -> comparación general
# df_folds                         -> métricas por fold
# reportes["Extra Trees"]           -> reporte por movimiento
# predicciones_oof["Extra Trees"]   -> predicciones por ventana
# matrices_confusion["Extra Trees"] -> matriz de confusión


COMPARACIÓN DE MODELOS
              modelo  accuracy_oof  precision_macro_oof  recall_macro_oof  f1_macro_oof  f1_weighted_oof  f1_macro_cv_media  f1_macro_cv_std  tiempo_segundos
HistGradientBoosting        0.9628               0.9618            0.9614        0.9612           0.9632             0.9614           0.0045         230.7004
         Extra Trees        0.9587               0.9583            0.9576        0.9572           0.9591             0.9572           0.0060           8.2549
       Random Forest        0.9525               0.9519            0.9513        0.9508           0.9529             0.9508           0.0067          90.3222
             SVM RBF        0.9424               0.9416            0.9417        0.9408           0.9429             0.9408           0.0074         210.5122
                 KNN        0.9286               0.9285            0.9274        0.9268           0.9295             0.9268           0.0030           7.6761
                 MLP        

In [8]:
print(reportes["HistGradientBoosting"])

              precision    recall  f1-score       support
0              0.991262  0.977909  0.984540   1856.000000
1              0.947593  0.916863  0.931975   1696.000000
10             0.970541  0.979904  0.975200   2488.000000
11             0.836440  0.954787  0.891704   1880.000000
12             0.913350  0.939846  0.926409   2344.000000
13             0.991857  0.972843  0.982258   2504.000000
15             0.989503  0.978834  0.984140   2504.000000
2              0.953067  0.960206  0.956623   2136.000000
3              0.978680  0.964000  0.971285   2000.000000
4              0.983644  0.942944  0.962864   2296.000000
5              0.987219  0.955631  0.971168   2344.000000
6              0.963768  0.959135  0.961446   2080.000000
7              0.988911  0.984416  0.986658   3080.000000
8              0.972539  0.962375  0.967430   2392.000000
9              0.958970  0.970684  0.964792   2456.000000
accuracy       0.962826  0.962826  0.962826      0.962826
macro avg     

In [9]:
print(matrices_confusion["HistGradientBoosting"])

Predicción     0     1    10    11    12    13    15     2     3     4     5  \
Real                                                                           
0           1815     3     4     3     0     0     0    11     5     0     0   
1              1  1555    23    57     0     4     0    38     4     0     0   
10             2    15  2438     2     2     0     0    15     6     0     0   
11             0     1     2  1795    72     0     0     0     0     4     3   
12             0     0     2    81  2203     1    12     0     1     0     0   
13             0     7     1     8    35  2436     7     0     2     3     1   
15             1     2     5     9    11     7  2451     1     0     0     0   
2              5    37     5    10     3     1     0  2051    15     0     3   
3              0    10    21     1     2     1     0    24  1928     0     1   
4              0     2     2    78    19     0     0     0     5  2165    16   
5              0     0     1    59    16

In [10]:
print(reportes["Extra Trees"])

              precision    recall  f1-score       support
0              0.990715  0.977371  0.983998   1856.000000
1              0.931846  0.910967  0.921288   1696.000000
10             0.941797  0.969051  0.955230   2488.000000
11             0.806034  0.994681  0.890476   1880.000000
12             0.975741  0.926621  0.950547   2344.000000
13             0.975971  0.973243  0.974605   2504.000000
15             0.987439  0.973243  0.980290   2504.000000
2              0.970106  0.941948  0.955819   2136.000000
3              0.960262  0.954500  0.957372   2000.000000
4              0.967235  0.938589  0.952697   2296.000000
5              0.986271  0.950085  0.967840   2344.000000
6              0.968193  0.965865  0.967028   2080.000000
7              0.980348  0.987987  0.984153   3080.000000
8              0.974968  0.944398  0.959439   2392.000000
9              0.957586  0.956026  0.956805   2456.000000
accuracy       0.958656  0.958656  0.958656      0.958656
macro avg     

In [11]:
print(reportes["Random Forest"])

              precision    recall  f1-score       support
0              0.986892  0.973599  0.980201   1856.000000
1              0.920097  0.896226  0.908005   1696.000000
10             0.931783  0.966238  0.948698   2488.000000
11             0.803349  0.995213  0.889047   1880.000000
12             0.962104  0.920648  0.940920   2344.000000
13             0.975080  0.968850  0.971955   2504.000000
15             0.985708  0.964058  0.974763   2504.000000
2              0.954826  0.940075  0.947393   2136.000000
3              0.959256  0.953500  0.956369   2000.000000
4              0.960289  0.926829  0.943262   2296.000000
5              0.982690  0.944539  0.963237   2344.000000
6              0.962062  0.950962  0.956480   2080.000000
7              0.982206  0.985714  0.983957   3080.000000
8              0.964255  0.936037  0.949936   2392.000000
9              0.947862  0.947476  0.947668   2456.000000
accuracy       0.952461  0.952461  0.952461      0.952461
macro avg     

In [13]:
print(reportes["SVM RBF"])

              precision    recall  f1-score       support
0              0.988643  0.984914  0.986775   1856.000000
1              0.914337  0.906250  0.910275   1696.000000
10             0.964026  0.958601  0.961306   2488.000000
11             0.778623  0.980319  0.867907   1880.000000
12             0.890363  0.859215  0.874512   2344.000000
13             0.959470  0.954872  0.957166   2504.000000
15             0.966653  0.960863  0.963749   2504.000000
2              0.940682  0.942884  0.941782   2136.000000
3              0.978517  0.956500  0.967383   2000.000000
4              0.948902  0.922038  0.935277   2296.000000
5              0.974775  0.923208  0.948291   2344.000000
6              0.899661  0.892308  0.895969   2080.000000
7              0.985197  0.972403  0.978758   3080.000000
8              0.978485  0.950669  0.964377   2392.000000
9              0.955429  0.960098  0.957758   2456.000000
accuracy       0.942418  0.942418  0.942418      0.942418
macro avg     

Movimiento 11 se confunde con 12, 4, 1 y 5